# Import the dependencies 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Import the dataset 

In [ ]:
aqi_dataset = pd.read_csv(r"comprehensive_indian_air_quality_2024.csv")
aqi_dataset.head()

## Check for the null values and the other information  

In [ ]:
aqi_dataset.info()

There are no null values and size of dataset is (3660,21).

## Checking the statistics of data

In [ ]:
aqi_dataset.describe()

Mean ≈ Median, therefore the distribution is approximately symmetric and shows little skewness

## Checking by visualising the data

In [ ]:
sns.pairplot(aqi_dataset.sample(500))

## Preparing Data for Modeling

In [ ]:
# clean column names
aqi_dataset.columns = (
    aqi_dataset.columns
    .str.strip()
    .str.replace(" ", "_")
)

# rename columns if necessary
aqi_dataset.rename(columns={
    "PM2_5": "PM2.5",
    "PM_10": "PM10",
    "CityType": "City_Type",
    "city_type": "City_Type",
    "Temp": "Temperature",
    "Hum": "Humidity"
}, inplace=True)

# required columns
required_cols = [
    "City_Type",
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "Temperature",
    "Humidity",
    "AQI"
]

# check missing columns
missing = [col for col in required_cols if col not in aqi_dataset.columns]

if len(missing) > 0:
    print("Missing Columns :", missing)
    print("Available Columns :")
    print(aqi_dataset.columns.tolist())
else:

    # keep required columns
    aqi_dataset = aqi_dataset[required_cols].copy()

    # convert city type to string
    aqi_dataset["City_Type"] = aqi_dataset["City_Type"].astype(str)

    # one hot encoding
    aqi_dataset = pd.get_dummies(
        aqi_dataset,
        columns=["City_Type"],
        drop_first=False
    )

    # feature list
    req_cols = [c for c in aqi_dataset.columns if c != "AQI"]

    print("Dataset Shape :", aqi_dataset.shape)
    print("Encoded Columns :")
    print([c for c in aqi_dataset.columns if "City_Type" in c])

Irrelevant columns were removed to keep only features that influence air quality.
Pollutant levels, temperature, and humidity were retained as predictors, while AQI was kept as the target variable.

The categorical feature City_Type was converted into numerical form using one-hot encoding so it could be used by the model.

After this step, the dataset contained only meaningful numeric features ready for training.

## Visualizing the required features

In [ ]:
pollution_features = [c for c in req_cols if not c.startswith("City_Type")]

print("Plotting features:", pollution_features)

sns.pairplot(aqi_dataset[pollution_features])

To better understand relationships between pollution and weather variables, the one-hot encoded City_Type columns were excluded from visualization.

Since these encoded columns contain only binary values (0/1), including them in pair plots can clutter the graphs and reduce interpretability.

### To understand the relationships between features, we visualize their correlations using a heatmap

In [ ]:
plt.figure(figsize=(10,10))

numeric_df = aqi_dataset[req_cols].astype(float)

sns.heatmap(numeric_df.corr().round(2), annot=True, cmap='coolwarm', fmt=".2f")

plt.title("Feature Correlation Heatmap")
plt.show()

Correlation analysis showed that major pollutants such as PM2.5, PM10, NO₂, SO₂, CO, and O₃ have a strong influence on AQI, so they were kept as the main predictors. Temperature and humidity were also retained due to their moderate impact on pollutant behaviour.

Features like wind speed, pressure, and calendar variables were removed since they showed weak or inconsistent relationships with AQI. The categorical variable City_Type was encoded to allow the model to capture regional differences in pollution patterns.

After these preprocessing steps, the dataset was ready for reliable model training and AQI prediction.

## Heatmap only for pollutants + AQI

In [ ]:
plt.figure(figsize=(10,10))

# keep only pollutant features + AQI
pollutant_cols = ['PM2.5','PM10','NO2','SO2','CO','O3','AQI']

pollutant_df = aqi_dataset[pollutant_cols].astype(float)

sns.heatmap(pollutant_df.corr().round(2),
            annot=True,
            cmap='coolwarm',
            fmt=".2f")

plt.title("Pollutant vs AQI Correlation Heatmap")
plt.show()

## Comparing Correlation Heatmaps

Two correlation heatmaps were generated for better analysis.
The first heatmap includes all features, including the one-hot encoded City_Type variables, to observe overall relationships in the dataset.

The second heatmap focuses only on pollutant variables along with AQI.
This filtered view removes categorical noise and highlights the direct environmental factors influencing air quality.

## Feature Skewness Check

We convert the selected columns to numeric values and compute their skewness.
This helps us understand the distribution of features and decide if transformations are needed before modeling.

In [ ]:
numeric_cols = [col for col in req_cols if col in aqi_dataset.columns]

numeric_df = aqi_dataset[numeric_cols].apply(pd.to_numeric, errors="coerce")

skew_vals = numeric_df.skew()

print(skew_vals)

## Visualizing Feature Skewness with Bar Plots

Bar plots are used to compare the skewness of individual features because they clearly show differences between discrete variables. Line plots are avoided since they suggest a continuous trend, which is not meaningful when comparing independent features.

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    x=skew_vals.index,
    y=skew_vals.values,
    color="steelblue"
)

plt.xticks(rotation=45, ha="right")
plt.title("Skewness of Numeric Features")
plt.xlabel("Features")
plt.ylabel("Skewness")

plt.tight_layout()
plt.show()

#### Before applying preprocessing steps, the dataset is divided into training and testing sets to avoid data leakage and ensure unbiased model evaluation.

In [ ]:
X = aqi_dataset.drop(columns=["AQI"])
Y = aqi_dataset["AQI"]

# Train-Test Split
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Reset indices
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
Y_train.reset_index(drop=True, inplace=True)
Y_test.reset_index(drop=True, inplace=True)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Target   :", Y_train.shape)
print("Testing Target    :", Y_test.shape)

Encode categorical columns after split and both datasets have same columns

With the dataset divided into training and testing sets, we now begin preprocessing.  
The initial step focuses on reducing skewness in the feature distributions to improve model performance.

## Handling Skewed Features

We identify highly skewed numerical features in the training data and apply a square-root transformation to reduce skewness.
If a feature contains negative values, it is shifted before transformation to ensure numerical stability.

This helps improve model performance by making feature distributions more balanced.

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns

continuous_cols = [
    col for col in numeric_cols
    if not col.startswith("City_Type")
]

# Calculate skewness
skew_vals = X_train[continuous_cols].skew()

# Columns with high skewness
skewed_cols = skew_vals[skew_vals.abs() > 0.75].index.tolist()

# Apply square-root transformation safely
for col in skewed_cols:

    train_min = X_train[col].min()
    test_min = X_test[col].min()
    min_value = min(train_min, test_min)

    if min_value < 0:
        shift = abs(min_value) + 1

        X_train[col] = np.sqrt(X_train[col] + shift)
        X_test[col] = np.sqrt(X_test[col] + shift)

    else:
        X_train[col] = np.sqrt(X_train[col])
        X_test[col] = np.sqrt(X_test[col])

print(f"Applied transformation to {len(skewed_cols)} columns")
print(skewed_cols)

## Checking the skewness of the features after handling

In [ ]:
# Convert all columns to numeric where possible
numeric_df = X_train.select_dtypes(include=["number"]).apply(
    pd.to_numeric,
    errors="coerce"
)

# Calculate skewness
af_skew_vals = numeric_df.skew().sort_values(ascending=False)

print("Skewness after preprocessing:")
print(af_skew_vals)

In [ ]:
numeric_train = X_train.apply(pd.to_numeric, errors='coerce')
skew_vals = numeric_train.skew()

plt.figure(figsize=(8,5))
sns.barplot(x=skew_vals.index, y=skew_vals.values, palette="coolwarm")

plt.xticks(rotation=90)
plt.xlabel('Features')
plt.ylabel('Skewness')
plt.title('Skewness of Numeric Features After Transformation')
plt.tight_layout()
plt.show()

## Using a pair plot to verify skewness after preprocessing.

In [ ]:
post_df = pd.concat([X_train, Y_train], axis=1)

env_cols = [
    'PM10','CO','Temperature','NO2',
    'SO2','O3','Humidity','PM2.5'
]

ordered_df = post_df[env_cols]

sns.pairplot(ordered_df, height=2.1)

The histograms indicate that skewness has been effectively reduced.  
We now apply StandardScaler to normalize feature ranges and improve model performance.

## Training the Random Forest Model

We train a Random Forest Regressor on the training data using the original (unscaled) feature values.
Since Random Forest is a tree-based model, it does not require feature scaling and can learn effectively from raw inputs.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor

identity = FunctionTransformer(func=None)

rf_pipeline = Pipeline([
    ("identity", identity),
    ("model", RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, Y_train)

rf_pipeline

## Feature Scaling

We standardize the features using StandardScaler before training the Linear Regression model.
Scaling is important because Linear Regression is sensitive to feature magnitudes, and normalization helps the model converge faster and assign balanced coefficients.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Numeric columns present in the dataset
numeric_features = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "Temperature",
    "Humidity"
]

# Keep only existing columns
numeric_features = [
    col for col in numeric_features
    if col in X_train.columns
]

# Create copies
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale numeric features
scaler = StandardScaler()

X_train_scaled.loc[:, numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled.loc[:, numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print("Scaled columns:", numeric_features)

After scaling, the dataset is now properly prepared and ready to be fed into the machine learning model.

## Using linear regression as model 1

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
import numpy as np

# Create model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train_scaled, Y_train)

# Predictions
train_pred = lr_model.predict(X_train_scaled)
test_pred = lr_model.predict(X_test_scaled)

print(f"Train R² Score : {r2_score(Y_train, train_pred):.4f}")
print(f"Test R² Score  : {r2_score(Y_test, test_pred):.4f}")
print(f"MAE            : {mean_absolute_error(Y_test, test_pred):.4f}")
print(f"RMSE           : {np.sqrt(mean_squared_error(Y_test, test_pred)):.4f}")

## Using random forest regressor as model 2

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

# Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, Y_train)

# Predictions
train_pred = rf_model.predict(X_train)
test_pred = rf_model.predict(X_test)


print(f"Train R² Score : {r2_score(Y_train, train_pred):.4f}")
print(f"Test R² Score  : {r2_score(Y_test, test_pred):.4f}")
print(f"MAE            : {mean_absolute_error(Y_test, test_pred):.4f}")
print(f"RMSE           : {np.sqrt(mean_squared_error(Y_test, test_pred)):.4f}")

Both Linear Regression and Random Forest models achieved high predictive performance, indicating that AQI is strongly determined by pollutant concentrations.  
Random Forest showed a slight improvement due to its ability to capture nonlinear relationships, but the overall increase was modest since the dataset already exhibits strong linear patterns.

## Model Evaluation and Overfitting Check

To ensure that the model generalizes well and does not memorize the training data,
we evaluate performance using:

1. Training score
2. Testing score
3. Cross-validation score

If the training score is much higher than testing or cross-validation scores,
the model may be overfitting. Otherwise, the model generalizes well.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("\nLinear Regression")
print(f"Train R² : {r2_score(Y_train, lr_train_pred):.4f}")
print(f"Test R²  : {r2_score(Y_test, lr_test_pred):.4f}")
print(f"MAE      : {mean_absolute_error(Y_test, lr_test_pred):.4f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(Y_test, lr_test_pred)):.4f}")

print("\nRandom Forest")
print(f"Train R² : {r2_score(Y_train, rf_train_pred):.4f}")
print(f"Test R²  : {r2_score(Y_test, rf_test_pred):.4f}")
print(f"MAE      : {mean_absolute_error(Y_test, rf_test_pred):.4f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(Y_test, rf_test_pred)):.4f}")

The training and testing R² scores are very close to each other,
indicating that the model performs similarly on both seen and unseen data.

This suggests that the model is not overfitting.

## Cross-Validation of the Model

We evaluate the Random Forest model using 5-fold cross-validation with the R² metric.
This helps assess how well the model generalizes to unseen data and detects possible overfitting.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

cv_scores = cross_val_score(
    rf_model,
    X_train,
    Y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("R² Scores :", np.round(cv_scores, 4))
print("Mean R²   :", round(cv_scores.mean(), 4))
print("Std Dev   :", round(cv_scores.std(), 4))
print(f"CV Score  : {cv_scores.mean() * 100:.2f}%")

The cross-validation score is also very close to the testing score,
confirming that the model performs consistently across different data splits.

This further indicates that the model generalizes well and is not overfitting.

## Saving the Trained Model

We create a pipeline containing the trained model and export it using pickle.
The feature column names are also saved to ensure consistency during future predictions or deployment.

In [ ]:
import pickle

# Train the final model (optional if already trained)
rf_model.fit(X_train, Y_train)

# Save the trained model
with open("aqi_model.pkl", "wb") as model_file:
    pickle.dump(rf_model, model_file)

# Save feature names
with open("columns.pkl", "wb") as column_file:
    pickle.dump(list(X_train.columns), column_file)

print("✅ Model and feature columns saved successfully!")

## Making a Sample Prediction

We create a sample input, apply the same encoding and feature alignment used during training, and load the saved pipeline to generate an AQI prediction.
This demonstrates how the trained model can be used for real-world inference.

In [ ]:
import pandas as pd
import pickle

# Load saved files
with open("aqi_model.pkl", "rb") as model_file:
    model = pickle.load(model_file)

with open("columns.pkl", "rb") as column_file:
    columns = pickle.load(column_file)

# Sample input
sample = pd.DataFrame({
    "PM2.5": [87.41],
    "PM10": [123.37],
    "NO2": [46.49],
    "SO2": [12.18],
    "CO": [1.16],
    "O3": [36.85],
    "Temperature": [18.20],
    "Humidity": [64.50],
    "City_Type": ["IT_hub"]
})

# One-hot encoding
sample = pd.get_dummies(sample, columns=["City_Type"])

# Match training columns
sample = sample.reindex(columns=columns, fill_value=0)

# Prediction
predicted_aqi = model.predict(sample)[0]

print(f"Predicted AQI: {predicted_aqi:.2f}")

## Early Warning System for Air Quality Risk

The system includes an ML-based early warning module that predicts the likelihood of unsafe air conditions. Instead of fixed thresholds, a classifier learns pollution patterns from sensor and environmental data to identify potential risk in advance.

In [ ]:
# Create Early Warning Target
# 0 -> AQI <= 200 (Safe/Moderate)
# 1 -> AQI > 200 (Poor/Very Poor/Severe)

aqi_dataset["Early_Risk"] = (
    aqi_dataset["AQI"] > 200
).astype(int)

# Display class distribution
print(aqi_dataset["Early_Risk"].value_counts())

# Percentage distribution
print("\nPercentage Distribution:")
print(
    aqi_dataset["Early_Risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

### Data Preprocessing
The dataset was preprocessed by separating features and target, applying one-hot encoding to categorical variables, converting all features to numeric values and filling missing values with the median to prepare the data for model training.

In [ ]:
import pandas as pd
import numpy as np

# Features
X_ew = aqi_dataset.drop(
    columns=[
        "AQI",
        "Early_Risk",
        "Risk_Level",
        "Early_Warning"
    ],
    errors="ignore"
)

# Target
y_ew = aqi_dataset["Early_Risk"].copy()

# One-Hot Encode categorical columns
categorical_cols = X_ew.select_dtypes(include=["object", "category"]).columns

if len(categorical_cols) > 0:
    X_ew = pd.get_dummies(
        X_ew,
        columns=categorical_cols,
        drop_first=False
    )

# Convert everything to numeric
X_ew = X_ew.apply(pd.to_numeric, errors="coerce")

# Fill missing values with median
X_ew = X_ew.fillna(X_ew.median())

print("Feature Shape :", X_ew.shape)
print("Target Shape  :", y_ew.shape)

### Random Forest Classifier
A Random Forest model was trained using an 80:20 train-test split (`random_state=42`) with balanced class weights. The model's performance was evaluated using Accuracy, Confusion Matrix and Classification Report (Precision, Recall, and F1-Score).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Train-Test Split
X_train_ew, X_test_ew, y_train_ew, y_test_ew = train_test_split(
    X_ew,
    y_ew,
    test_size=0.20,
    random_state=42,
    stratify=y_ew
)

# Create Model
early_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train
early_model.fit(X_train_ew, y_train_ew)

# Predictions
y_pred = early_model.predict(X_test_ew)

# Evaluation
print("=" * 50)
print("Early Warning System Performance")
print("=" * 50)

print(f"Accuracy : {accuracy_score(y_test_ew, y_pred):.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ew, y_pred))

print("\nClassification Report")
print(classification_report(y_test_ew, y_pred))

In [ ]:
import pickle

# Save Early Warning Model
with open("early_model.pkl", "wb") as model_file:
    pickle.dump(early_model, model_file)

# Save Feature Names
with open("early_columns.pkl", "wb") as column_file:
    pickle.dump(list(X_ew.columns), column_file)

print("Early Warning model saved successfully!")
print("early_model.pkl created")
print("early_columns.pkl created")

## Some important plots / graphs


#### Actual vs Predicted AQI Plot

In [ ]:
import matplotlib.pyplot as plt

# Actual vs Predicted AQI
plt.figure(figsize=(7, 6))

plt.scatter(
    Y_test,
    rf_test_pred,
    alpha=0.7,
    edgecolors="black"
)

# Perfect prediction line
min_val = min(Y_test.min(), rf_test_pred.min())
max_val = max(Y_test.max(), rf_test_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    color="red",
    label="Perfect Prediction"
)

plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Random Forest: Actual vs Predicted AQI")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

This plot compares actual AQI values with those predicted by the Random Forest model. Points closer to the diagonal line indicate more accurate predictions, while distant points show larger errors. It provides a visual check of how well the model performs on unseen data.

#### Residual Plot (Random Forest)

In [ ]:
residuals = Y_test - rf_test_pred

plt.figure(figsize=(7, 5))

plt.scatter(
    rf_test_pred,
    residuals,
    alpha=0.7,
    edgecolors="black"
)

# Reference line
plt.axhline(
    y=0,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Zero Error"
)

plt.xlabel("Predicted AQI")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Random Forest Residual Plot")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

This residual plot shows the difference between actual and predicted AQI values against the model’s predictions. A random scatter around zero indicates good model performance, while visible patterns may suggest missing relationships or bias in the model.

#### Feature Importance Plot (from RF)

In [ ]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

# Plot
plt.figure(figsize=(9, 6))

feature_importance.plot(
    kind="barh",
    color="steelblue"
)

plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.title("Random Forest Feature Importance")

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.show()

This plot shows how much each feature contributes to the AQI prediction in the Random Forest model. Features with higher importance scores have a stronger influence on the model’s decisions, while features with very low scores contribute little to the prediction.

From this graph, the most influential pollutants affecting AQI can be easily identified.

### AQI Distribution Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    data=aqi_dataset,
    x="AQI",
    bins=30,
    kde=True,
    color="steelblue"
)

plt.title("Distribution of AQI Values")
plt.xlabel("AQI")
plt.ylabel("Count")

plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

This histogram shows how AQI values are distributed across the dataset. It helps identify common pollution ranges as well as extreme AQI levels. Understanding this distribution is useful for detecting skewness, outliers, and overall air quality patterns in the data.